Задание 1. Моделирование стоимости I/O (Бенчмарк дисковой подсистемы)

Суть задания:

Цель этой практики — разрушить иллюзию о том, что чтение 100 Мегабайт данных
всегда занимает одинаковое время. Мы на практике проверим, как механика работы
дисковых накопителей (и подсистемы ввода-вывода ОС) влияет на производительность
приложения при последовательном и случайном доступе (Sequential vs. Random Access).

Что происходит в коде:

Скрипт состоит из трех логических блоков:

generate_test_file: Создает бинарный файл-заглушку (по умолчанию 100 МБ), заполненный
случайными байтами. Мы пишем его блоками по 4 КБ — это размер стандартной страницы памяти в большинстве ОС.
sequential_read_test: Имитирует типичную I/O-bound нагрузку вроде Full Table Scan в базе
данных или чтения лога от начала до конца. Мы открываем файл и читаем его непрерывным потоком.
В этот момент операционная система понимает наш паттерн чтения и включает упреждающее чтение (read-ahead) — заранее подтягивает следующие блоки с диска в кэш ОС (Page Cache).

random_read_test: Имитирует нагрузку, характерную для плохо спроектированных индексов или
фрагментированных таблиц. Мы читаем ровно тот же объем данных (100 МБ), но делаем это хаотично.
Скрипт случайным образом вычисляет адрес блока внутри файла, прыгает туда курсором (симуляция seek)
и считывает 4 КБ. Операционная система не может предсказать наш следующий шаг,
prefetching становится бесполезным, и каждое чтение превращается в дорогой физический I/O запрос.
Что требуется сделать студенту (разбор TODO):

В TODO 1.1 нужно реализовать классический цикл чтения файла кусками (chunks) до тех пор,
пока не будет достигнут конец файла (EOF). Важно правильно инкрементировать
счетчик прочитанных байт, чтобы убедиться, что мы вычитали весь объем.

В TODO 1.2 нужно реализовать логику «прыжков» по файлу. Студент должен
сгенерировать случайный номер блока, перевести его в байтовое смещение
(умножив на размер блока) и использовать метод f.seek(offset) перед чтением.
Ожидаемый результат: При запуске скрипта время выполнения random_read_test должно
оказаться в десятки, если не в сотни раз больше, чем sequential_read_test.
Разрыв будет особенно показательным, если запустить это на классическом HDD.


In [2]:
import os
import time
import random


FILE_NAME = "io_benchmark_dummy.dat"
FILE_SIZE_MB = 100  # 100 МБ для быстрого теста, на мощных машинах можно поднять до 500
BLOCK_SIZE = 4096   # 4 КБ - стандартный размер страницы/блока ОС


def generate_test_file():
    """Создает тестовый файл нужного размера, если его нет."""
    if os.path.exists(FILE_NAME) and os.path.getsize(FILE_NAME) == FILE_SIZE_MB * 1024 * 1024:
        print("Файл уже существует, пропускаем генерацию.")
        return

    print(f"Генерация тестового файла ({FILE_SIZE_MB} MB)...")
    with open(FILE_NAME, "wb") as f:
        # Пишем блоками для скорости
        for _ in range((FILE_SIZE_MB * 1024 * 1024) // BLOCK_SIZE):
            f.write(os.urandom(BLOCK_SIZE))
    print("Готово.\n")


def sequential_read_test():
    """Тест последовательного чтения - читаем файл блоками от начала до конца."""
    print("Запуск последовательного чтения...")
    start_time = time.time()

    bytes_read = 0
    with open(FILE_NAME, "rb") as f:
        # TODO 1.1: Напишите цикл (while), который читает файл непрерывно по BLOCK_SIZE
        # до тех пор, пока не достигнет конца файла (EOF).
        # Обновляйте счетчик bytes_read.
        while True:
            data = f.read(BLOCK_SIZE)
            bytes_read += len(data)
            if not data:
                break

    elapsed = time.time() - start_time
    print(f"Последовательное чтение: прочитано {bytes_read / 1024 / 1024:.2f} MB за {elapsed:.4f} сек.")
    return elapsed


def random_read_test():
    """Тест случайного чтения - читаем тот же объем данных, но прыгаем по файлу."""
    print("Запуск случайного чтения (seek)...")
    total_blocks = (FILE_SIZE_MB * 1024 * 1024) // BLOCK_SIZE

    start_time = time.time()
    bytes_read = 0
    with open(FILE_NAME, "rb") as f:
        # TODO 1.2: Напишите цикл, который выполнится total_blocks раз.
        # Внутри цикла:
        # 1. Сгенерируйте случайный индекс блока: random.randint(0, total_blocks - 1)
        # 2. Вычислите смещение (offset) в байтах.
        # 3. Переместите курсор файла с помощью f.seek()
        # 4. Прочитайте BLOCK_SIZE байт и обновите счетчик bytes_read
        for _ in range(total_blocks):
            idx = random.randint(0, total_blocks - 1)
            offset = BLOCK_SIZE * idx
            f.seek(offset)
            data = f.read(BLOCK_SIZE)
            bytes_read += len(data)

    elapsed = time.time() - start_time
    print(f"Случайное чтение: прочитано {bytes_read / 1024 / 1024:.2f} MB за {elapsed:.4f} сек.")
    return elapsed


if __name__ == "__main__":
    generate_test_file()

    seq_time = sequential_read_test()
    rand_time = random_read_test()

    if seq_time > 0:
        print(f"\nВывод: Случайное чтение медленнее последовательного в {rand_time / seq_time:.2f} раз(а)")

Файл уже существует, пропускаем генерацию.
Запуск последовательного чтения...
Последовательное чтение: прочитано 100.00 MB за 0.0264 сек.
Запуск случайного чтения (seek)...
Случайное чтение: прочитано 100.00 MB за 0.0467 сек.

Вывод: Случайное чтение медленнее последовательного в 1.77 раз(а)
